# EEGNet LOSO Export Notebook

This notebook builds the **missing export layer** required for EEGNet LOSO interpretability.

## Purpose

The downstream LOSO interpretability notebook expects saved per-subject EEGNet artifacts.
At the moment, those artifacts do not exist, so discovery fails before any attribution step begins.

This notebook fixes that by exporting, for each held-out subject and each seed:

- a trained EEGNet checkpoint
- train-derived normalization statistics
- the held-out subject evaluation cache
- held-out probabilities and predictions
- training history
- subject-level metrics
- a manifest for later discovery

## Design principles

This notebook is intentionally separate from the main LOSO benchmark notebook.

- The main LOSO notebook should remain focused on benchmarking and summary analysis.
- This notebook is focused only on reproducible export for later interpretability.
- The export format is designed to match the needs of the downstream interpretability workflow.

## Important note

Because EEGNet is trained with train-only normalization, the export must include
the normalization mean and standard deviation used for the held-out subject model.
Without that, later attribution would not be applied to the model under the same
input scaling used during training.

In [ ]:
# Run from the repository root so that relative result paths resolve
# identically regardless of where this notebook lives.
from oddball_eeg.paths import use_project_root

PROJECT_ROOT = use_project_root()

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn

from oddball_eeg.bids import make_epoching_config_from_experiment
from oddball_eeg.config import get_experiment
from oddball_eeg.io import save_dataframe, save_experiment_metadata, save_json
from oddball_eeg.loader import load_subjects_grouped
from oddball_eeg.models import get_model
from oddball_eeg.preprocessing import (
    apply_crop_to_loaded_dataset,
    build_dataloaders,
    build_datasets_from_split,
    make_class_weight_tensor,
    SplitArrays,
)
from oddball_eeg.reproducibility import get_default_device, seed_everything
from oddball_eeg.splits import make_group_train_val_split
from oddball_eeg.training import evaluate_eegnet_loader
from oddball_eeg.visualization import (
    plot_learning_curve,
    round_numeric_columns,
    save_figure,
    set_pandas_display_options,
)


set_pandas_display_options()

plt.rcParams["figure.dpi"] = 140
plt.rcParams["savefig.dpi"] = 300

DEVICE = get_default_device()

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DEVICE      :", DEVICE)

## Configuration

This notebook uses the frozen `loso_main` experiment configuration from the project.

It also applies the same crop window used by the main thesis pipeline so that the
exported held-out tensors match the LOSO setting already used in the benchmark.

In [ ]:
# EXPERIMENT_NAME = "ds005863_loso_main"
EXPERIMENT_NAME = "ds003061_loso_main"

cfg = get_experiment(EXPERIMENT_NAME)

EXPORT_ROOT = cfg.artifact_dir("eegnet_loso_exports")
FIG_ROOT = cfg.artifact_dir("eegnet_loso_exports", "figures")
TABLE_ROOT = cfg.artifact_dir("eegnet_loso_exports", "tables")
CHECKPOINT_ROOT = cfg.checkpoint_dir("eegnet_loso_exports")

EXPORT_ROOT.mkdir(parents=True, exist_ok=True)
FIG_ROOT.mkdir(parents=True, exist_ok=True)
TABLE_ROOT.mkdir(parents=True, exist_ok=True)
CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)

EEGNET_KWARGS = {
    "n_classes": cfg.eegnet.n_classes,
    "f1": cfg.eegnet.f1,
    "d": cfg.eegnet.d,
    "f2": cfg.eegnet.f2,
    "k1": cfg.eegnet.k1,
    "k2": cfg.eegnet.k2,
    "dropout": cfg.eegnet.dropout,
}

CONFIG_SNAPSHOT = {
    "experiment_key": EXPERIMENT_NAME,
    "experiment_name": cfg.name,
    "dataset": cfg.dataset.dataset,
    "task": cfg.dataset.task,
    "subjects": list(cfg.subjects.subjects),
    "seeds": list(cfg.seeds.seeds),
    "signal": cfg.signal.slug(),
    "crop_window": cfg.effective_crop_window(),
    "eegnet_kwargs": EEGNET_KWARGS,
    "normalize": cfg.eegnet.normalize,
    "batch_size": cfg.eegnet.batch_size,
    "lr": cfg.eegnet.lr,
    "weight_decay": cfg.eegnet.weight_decay,
    "epochs": cfg.eegnet.epochs,
    "early_stopping_patience": cfg.eegnet.early_stopping_patience,
    "val_size": cfg.splits.val_size,
    "val_n_tries": cfg.splits.val_n_tries,
    "export_root": str(EXPORT_ROOT),
    "checkpoint_root": str(CHECKPOINT_ROOT),
}

save_experiment_metadata(
    cfg,
    extra={
        "notebook": "12_eegnet_loso_export.ipynb",
        "purpose": "Export LOSO EEGNet checkpoints, normalization stats, held-out caches, predictions, histories, and manifest.",
        "experiment_key": EXPERIMENT_NAME,
        "export_root": str(EXPORT_ROOT),
        "checkpoint_root": str(CHECKPOINT_ROOT),
    },
)

print("Experiment key :", EXPERIMENT_NAME)
print("Experiment name:", cfg.name)
print("Dataset        :", cfg.dataset.dataset)
print("Task           :", cfg.dataset.task)
print("Subjects       :", cfg.subjects.subjects)
print("Seeds          :", cfg.seeds.seeds)
print("Signal         :", cfg.signal.slug())
print("Crop           :", cfg.effective_crop_window())
print("Normalize      :", cfg.eegnet.normalize)
print("Export root    :", EXPORT_ROOT)
print("Checkpoint root:", CHECKPOINT_ROOT)

## Load the grouped chronological dataset

We reuse the existing project loader so the exported arrays stay aligned with the
same chronological reconstruction and channel/time consistency checks already used
elsewhere in the thesis pipeline.

In [ ]:
epoching_cfg = make_epoching_config_from_experiment(cfg)

print("Epoching config:")
print(epoching_cfg)

data_full = load_subjects_grouped(
    dataset=cfg.dataset.dataset,
    task_name=cfg.dataset.task,
    subjects=cfg.subjects.subjects,
    epoching_cfg=epoching_cfg,
    chunk_size=cfg.splits.group_chunk_size,
    apply_baseline=True,
    verbose=True,
)

crop_window = cfg.effective_crop_window()

if crop_window is not None:
    data = apply_crop_to_loaded_dataset(
        data_full,
        tmin=crop_window[0],
        tmax=crop_window[1],
    )
else:
    data = data_full

qc_rows = []

for sub in data.subjects:
    X_sub = data.X_by_sub[sub]
    y_sub = data.y_by_sub[sub]

    qc_rows.append({
        "subject": sub,
        "n_trials": int(X_sub.shape[0]),
        "n_channels": int(X_sub.shape[1]),
        "n_times": int(X_sub.shape[2]),
        "n_standard": int((y_sub == 0).sum()),
        "n_target": int((y_sub == 1).sum()),
        "target_rate": float((y_sub == 1).mean()),
        "n_groups": int(pd.Series(data.groups_by_sub[sub]).nunique()),
    })

qc_df = (
    pd.DataFrame(qc_rows)
    .sort_values("subject")
    .reset_index(drop=True)
)

save_dataframe(
    qc_df,
    TABLE_ROOT / "loso_export_dataset_qc.csv",
)

display(round_numeric_columns(qc_df, 4))

## LOSO helpers

The benchmark notebook already defines LOSO logic locally.
We reproduce the minimal helper layer here so the export notebook stays self-contained.

In [ ]:
def concatenate_training_subjects_with_unique_groups(data, train_subjects):
    """
    Concatenate all training subjects into one training set while offsetting
    group IDs so they remain unique across subjects.
    """
    X_list, y_list, g_list = [], [], []
    group_offset = 0

    for sub in train_subjects:
        X_sub = data.X_by_sub[sub]
        y_sub = data.y_by_sub[sub]
        g_sub = data.groups_by_sub[sub]

        g_sub_unique = g_sub + group_offset

        X_list.append(X_sub)
        y_list.append(y_sub)
        g_list.append(g_sub_unique)

        group_offset = int(g_sub_unique.max()) + 1

    X_train = np.concatenate(X_list, axis=0).astype(np.float32)
    y_train = np.concatenate(y_list, axis=0).astype(np.int64)
    g_train = np.concatenate(g_list, axis=0).astype(np.int64)

    return X_train, y_train, g_train


def make_seed_subject_dir(export_root: Path, seed: int, subject: str) -> Path:
    out = export_root / f"seed_{seed:03d}" / f"sub-{subject}"
    out.mkdir(parents=True, exist_ok=True)
    return out


def apply_norm_numpy(X: np.ndarray, mean: np.ndarray, std: np.ndarray, eps: float = 1e-8) -> np.ndarray:
    return ((X - mean) / (std + eps)).astype(np.float32)


@torch.no_grad()
def predict_eegnet_prob(model: nn.Module, X: np.ndarray, mean: np.ndarray, std: np.ndarray, device) -> np.ndarray:
    """
    Predict class-1 probabilities using the same train-derived normalization
    convention used during EEGNet training.
    """
    model.eval()

    Xn = apply_norm_numpy(X, mean=mean, std=std)

    probs = []
    batch_size = 256

    for start in range(0, len(Xn), batch_size):
        xb = torch.from_numpy(Xn[start:start + batch_size]).float().unsqueeze(1).to(device)
        logits = model(xb)

        if logits.ndim == 2 and logits.shape[1] >= 2:
            prob1 = torch.softmax(logits, dim=1)[:, 1].detach().cpu().numpy()
        elif logits.ndim == 2 and logits.shape[1] == 1:
            prob1 = torch.sigmoid(logits[:, 0]).detach().cpu().numpy()
        elif logits.ndim == 1:
            prob1 = torch.sigmoid(logits).detach().cpu().numpy()
        else:
            raise ValueError(f"Unexpected EEGNet output shape: {tuple(logits.shape)}")

        probs.append(prob1)

    return np.concatenate(probs, axis=0)

def build_checkpoint_payload(
        model: nn.Module,
        *,
        subject: str,
        seed: int,
        train_subjects: list[str],
        best_epoch: int | None,
        best_val_auc: float,
        mean: np.ndarray,
        std: np.ndarray,
):
    return {
        "state_dict": {k: v.detach().cpu() for k, v in model.state_dict().items()},
        "model_name": "eegnet",
        "model_kwargs": dict(EEGNET_KWARGS),
        "subject": str(subject),
        "seed": int(seed),
        "train_subjects": [str(s) for s in train_subjects],
        "test_subject": str(subject),
        "best_epoch": None if best_epoch is None else int(best_epoch),
        "best_val_auc": float(best_val_auc),
        "normalize": bool(cfg.eegnet.normalize),
        "norm_mean": torch.as_tensor(mean, dtype=torch.float32).cpu(),
        "norm_std": torch.as_tensor(std, dtype=torch.float32).cpu(),
        "crop_window": None if cfg.effective_crop_window() is None else tuple(float(x) for x in cfg.effective_crop_window()),
        "dataset": str(cfg.dataset.dataset),
        "task": str(cfg.dataset.task),
        "signal": cfg.signal.slug(),
        "n_channels": int(mean.shape[1]) if mean.ndim == 3 else None,
    }

## Train-one-subject-and-export function

This function trains LOSO EEGNet for one held-out subject, then exports everything
needed later for interpretability and report-ready reuse.

In [ ]:
def export_eegnet_loso_one_subject(
        *,
        data,
        cfg,
        test_subject: str,
        seed: int,
        export_root: Path,
        checkpoint_root: Path,
        device,
        verbose: bool = True,
):
    seed_everything(seed)

    train_subjects = [s for s in data.subjects if s != test_subject]

    X_train_all, y_train_all, g_train_all = concatenate_training_subjects_with_unique_groups(
        data=data,
        train_subjects=train_subjects,
    )

    X_test = data.X_by_sub[test_subject].astype(np.float32)
    y_test = data.y_by_sub[test_subject].astype(np.int64)
    g_test = data.groups_by_sub[test_subject].astype(np.int64)
    sample_idx_test = data.samples_by_sub[test_subject].astype(np.int64)

    tr_sub_idx, val_sub_idx = make_group_train_val_split(
        y_train=y_train_all,
        groups_train=g_train_all,
        val_size=cfg.splits.val_size,
        seed=seed,
        n_tries=cfg.splits.val_n_tries,
    )

    split = SplitArrays(
        X_train=X_train_all[tr_sub_idx],
        y_train=y_train_all[tr_sub_idx],
        X_val=X_train_all[val_sub_idx],
        y_val=y_train_all[val_sub_idx],
        X_test=X_test,
        y_test=y_test,
    )

    ds_train, ds_val, ds_test, norm_stats = build_datasets_from_split(
        split,
        normalize=cfg.eegnet.normalize,
    )

    train_loader, val_loader, test_loader = build_dataloaders(
        ds_train=ds_train,
        ds_val=ds_val,
        ds_test=ds_test,
        batch_size=cfg.eegnet.batch_size,
        device=device,
    )

    model = get_model(
        "eegnet",
        n_ch=split.X_train.shape[1],
        n_times=split.X_train.shape[2],
        eegnet_kwargs=dict(EEGNET_KWARGS),
    ).to(device)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=cfg.eegnet.lr,
        weight_decay=cfg.eegnet.weight_decay,
    )

    if cfg.eegnet.use_class_weight:
        class_weight = make_class_weight_tensor(split.y_train, device=device)
        criterion = nn.CrossEntropyLoss(weight=class_weight)
    else:
        criterion = nn.CrossEntropyLoss()

    history_rows = []
    best_val_auc = -np.inf
    best_epoch = None
    best_state = None
    patience_left = cfg.eegnet.early_stopping_patience

    for ep in range(1, cfg.eegnet.epochs + 1):
        model.train()
        batch_losses = []

        for xb, yb in train_loader:
            xb = xb.unsqueeze(1).to(device)
            yb = yb.to(device)

            logits = model(xb)
            loss = criterion(logits, yb)

            optimizer.zero_grad(set_to_none=True)
            loss.backward()

            if cfg.eegnet.grad_clip_norm is not None and cfg.eegnet.grad_clip_norm > 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.eegnet.grad_clip_norm)

            optimizer.step()
            batch_losses.append(float(loss.item()))

        train_loss = float(np.mean(batch_losses)) if batch_losses else float("nan")
        val_metrics = evaluate_eegnet_loader(model, val_loader, device=device)

        history_rows.append({
            "epoch": int(ep),
            "train_loss": train_loss,
            "val_loss": float(val_metrics["loss"]),
            "val_auc": float(val_metrics["auc"]),
            "val_pr_auc": float(val_metrics["pr_auc"]),
            "val_bal_acc": float(val_metrics["bal_acc"]),
        })

        if verbose and (ep == 1 or ep % 5 == 0 or ep == cfg.eegnet.epochs):
            print(
                f"[sub-{test_subject} | seed {seed}] "
                f"epoch={ep:03d} "
                f"train_loss={train_loss:.4f} "
                f"val_auc={val_metrics['auc']:.4f}"
            )

        if np.isfinite(val_metrics["auc"]) and val_metrics["auc"] > best_val_auc + 1e-4:
            best_val_auc = float(val_metrics["auc"])
            best_epoch = int(ep)
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            patience_left = cfg.eegnet.early_stopping_patience
        else:
            patience_left -= 1
            if patience_left <= 0:
                if verbose:
                    print(f"[sub-{test_subject} | seed {seed}] early stopping at epoch {ep}")
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    test_metrics = evaluate_eegnet_loader(model, test_loader, device=device)

    mean = norm_stats.mean.astype(np.float32)
    std = norm_stats.std.astype(np.float32)

    prob = predict_eegnet_prob(
        model=model,
        X=X_test,
        mean=mean,
        std=std,
        device=device,
    )

    pred = (prob >= 0.5).astype(np.int64)
    correct = (pred == y_test).astype(np.int64)

    out_dir = make_seed_subject_dir(export_root, seed=seed, subject=test_subject)
    ckpt_dir = make_seed_subject_dir(checkpoint_root, seed=seed, subject=test_subject)

    checkpoint_path = ckpt_dir / "checkpoint.pt"
    heldout_eval_path = out_dir / "heldout_eval.npz"
    history_path = out_dir / "history.csv"
    metrics_path = out_dir / "metrics.json"
    learning_curve_path = out_dir / "learning_curve.png"

    ckpt_payload = build_checkpoint_payload(
        model=model,
        subject=test_subject,
        seed=seed,
        train_subjects=train_subjects,
        best_epoch=best_epoch,
        best_val_auc=best_val_auc,
        mean=mean,
        std=std,
    )

    torch.save(ckpt_payload, checkpoint_path)

    np.savez_compressed(
        heldout_eval_path,
        X=X_test.astype(np.float32),
        y=y_test.astype(np.int64),
        subjects=np.asarray([test_subject] * len(X_test), dtype=str),
        groups=g_test.astype(np.int64),
        sample_idx=sample_idx_test.astype(np.int64),
        times=np.asarray(data.times, dtype=np.float32),
        ch_names=np.asarray(data.ch_names, dtype=str),
        sfreq=np.asarray(float(data.sfreq), dtype=np.float32),
        prob=prob.astype(np.float32),
        pred=pred.astype(np.int64),
        correct=correct.astype(np.int64),
    )

    history_df = pd.DataFrame(history_rows)
    save_dataframe(history_df, history_path)

    if len(history_df) > 0:
        fig = plot_learning_curve(
            history_df,
            title=f"LOSO EEGNet learning curve — sub-{test_subject} — seed {seed}",
        )
        save_figure(fig, learning_curve_path)
        plt.close(fig)

    metrics = {
        "subject": test_subject,
        "seed": int(seed),
        "train_subjects": train_subjects,
        "n_train_total": int(len(X_train_all)),
        "n_train_used": int(len(split.X_train)),
        "n_val_used": int(len(split.X_val)),
        "n_test": int(len(X_test)),
        "n_channels": int(X_test.shape[1]),
        "n_times": int(X_test.shape[2]),
        "best_epoch": None if best_epoch is None else int(best_epoch),
        "best_val_auc": float(best_val_auc),
        "test_auc": float(test_metrics["auc"]),
        "test_pr_auc": float(test_metrics["pr_auc"]),
        "test_acc": float(test_metrics["acc"]),
        "test_bal_acc": float(test_metrics["bal_acc"]),
        "test_f1": float(test_metrics["f1"]),
        "tn": int(test_metrics["tn"]),
        "fp": int(test_metrics["fp"]),
        "fn": int(test_metrics["fn"]),
        "tp": int(test_metrics["tp"]),
        "mean_prob": float(np.mean(prob)),
        "target_rate": float(np.mean(y_test)),
        "n_correct": int(correct.sum()),
        "n_incorrect": int((1 - correct).sum()),
        "checkpoint_path": str(checkpoint_path.resolve()),
        "heldout_eval_path": str(heldout_eval_path.resolve()),
        "history_path": str(history_path.resolve()),
        "learning_curve_path": str(learning_curve_path.resolve()),
        "metrics_path": str(metrics_path.resolve()),
    }

    save_json(metrics, metrics_path)

    return metrics

## Run all exports

This may take a while because it trains one LOSO EEGNet per held-out subject per seed.
The output is a manifest table that later notebooks can use directly.

In [ ]:
all_metric_rows = []

for seed in cfg.seeds.seeds:
    print("\n" + "#" * 90)
    print(f"SEED {seed}")
    print("#" * 90)

    for test_subject in data.subjects:
        print("\n" + "=" * 80)
        print(f"EXPORTING HELD-OUT SUBJECT sub-{test_subject}")
        print("=" * 80)

        row = export_eegnet_loso_one_subject(
            data=data,
            cfg=cfg,
            test_subject=test_subject,
            seed=seed,
            export_root=EXPORT_ROOT,
            checkpoint_root=CHECKPOINT_ROOT,
            device=DEVICE,
            verbose=cfg.verbose,
        )

        all_metric_rows.append(row)

manifest_df = (
    pd.DataFrame(all_metric_rows)
    .sort_values(["seed", "subject"])
    .reset_index(drop=True)
)

save_dataframe(
    manifest_df,
    EXPORT_ROOT / "manifest.csv",
)

save_json(
    CONFIG_SNAPSHOT,
    EXPORT_ROOT / "config_snapshot.json",
)

save_experiment_metadata(
    cfg,
    extra={
        "notebook": "12_eegnet_loso_export.ipynb",
        "status": "completed",
        "manifest_path": str(EXPORT_ROOT / "manifest.csv"),
        "config_snapshot_path": str(EXPORT_ROOT / "config_snapshot.json"),
        "n_exports": int(len(manifest_df)),
    },
)

print("Done.")
display(round_numeric_columns(manifest_df, 4))

## Quick export validation

This checks that the expected files were actually created.

In [ ]:
check_rows = []

for _, row in manifest_df.iterrows():
    ckpt = Path(row["checkpoint_path"])
    npz = Path(row["heldout_eval_path"])
    hist = Path(row["history_path"])
    metrics = Path(row["metrics_path"])

    check_rows.append({
        "seed": int(row["seed"]),
        "subject": row["subject"],
        "checkpoint_exists": ckpt.exists(),
        "npz_exists": npz.exists(),
        "history_exists": hist.exists(),
        "metrics_exists": metrics.exists(),
    })

check_df = (
    pd.DataFrame(check_rows)
    .sort_values(["seed", "subject"])
    .reset_index(drop=True)
)

save_dataframe(
    check_df,
    TABLE_ROOT / "loso_export_file_check.csv",
)

display(check_df)

assert check_df[["checkpoint_exists", "npz_exists", "history_exists", "metrics_exists"]].all().all(), \
    "Some LOSO export files are missing."

## Inspect one exported subject package

In [ ]:
example_row = manifest_df.iloc[0]

example_seed = int(example_row["seed"])
example_subject = str(example_row["subject"])

example_ckpt_path = Path(example_row["checkpoint_path"])
example_npz_path = Path(example_row["heldout_eval_path"])
example_history_path = Path(example_row["history_path"])
example_metrics_path = Path(example_row["metrics_path"])

print("Example seed   :", example_seed)
print("Example subject:", example_subject)
print("Checkpoint     :", example_ckpt_path)
print("Heldout NPZ    :", example_npz_path)
print("History        :", example_history_path)
print("Metrics        :", example_metrics_path)

print("\nFile exists check:")
print("checkpoint:", example_ckpt_path.exists())
print("npz       :", example_npz_path.exists())
print("history   :", example_history_path.exists())
print("metrics   :", example_metrics_path.exists())

assert example_ckpt_path.exists(), f"Missing checkpoint: {example_ckpt_path}"
assert example_npz_path.exists(), f"Missing heldout cache: {example_npz_path}"
assert example_history_path.exists(), f"Missing history: {example_history_path}"
assert example_metrics_path.exists(), f"Missing metrics: {example_metrics_path}"

# trusted local file -> allow full load
example_ckpt = torch.load(
    example_ckpt_path,
    map_location="cpu",
    weights_only=False,
)

example_npz = np.load(example_npz_path, allow_pickle=True)
example_history = pd.read_csv(example_history_path)

print("\nCheckpoint keys:")
print(sorted(example_ckpt.keys()))

print("\nNPZ keys:")
print(example_npz.files)

print("\nHistory head:")
display(example_history.head())

## Optional compact summary

This is useful for checking subject difficulty and for deciding which seed to use
first in the downstream interpretability notebook.

In [ ]:
summary_df = (
    manifest_df.groupby("subject", as_index=False)
    .agg(
        n_seeds=("seed", "count"),
        auc_mean=("test_auc", "mean"),
        auc_std=("test_auc", "std"),
        bal_acc_mean=("test_bal_acc", "mean"),
        best_auc=("test_auc", "max"),
        worst_auc=("test_auc", "min"),
    )
    .sort_values("subject")
    .reset_index(drop=True)
)

save_dataframe(
    summary_df,
    TABLE_ROOT / "loso_export_subject_summary.csv",
)

manifest = {
    "experiment_key": EXPERIMENT_NAME,
    "dataset": cfg.dataset.dataset,
    "task": cfg.dataset.task,
    "signal": cfg.signal.slug(),
    "export_root": str(EXPORT_ROOT),
    "checkpoint_root": str(CHECKPOINT_ROOT),
    "manifest_csv": str(EXPORT_ROOT / "manifest.csv"),
    "config_snapshot_json": str(EXPORT_ROOT / "config_snapshot.json"),
    "n_rows": int(len(manifest_df)),
    "subjects": sorted(manifest_df["subject"].astype(str).unique().tolist()),
    "seeds": sorted([int(x) for x in manifest_df["seed"].unique().tolist()]),
}

save_json(
    manifest,
    EXPORT_ROOT / "manifest.json",
)

print("Export root:", EXPORT_ROOT)
print("Manifest   :", EXPORT_ROOT / "manifest.csv")
print("Checkpoint :", CHECKPOINT_ROOT)

display(round_numeric_columns(summary_df, 4))

print("\nCreated files:")
for p in sorted(EXPORT_ROOT.rglob("*")):
    if p.is_file():
        print("-", p.relative_to(EXPORT_ROOT))

## Final note for the downstream interpretability notebook

When you proceed to Notebook 13, update its discovery logic to use the manifest saved by this notebook:

- cfg_loso.artifact_dir("eegnet_loso_exports") / "manifest.csv"

Notebook 13 should load checkpoint_path and heldout_eval_path directly from the manifest instead of reconstructing paths manually.

In [ ]:
print("Export root:", EXPORT_ROOT)
print("\nCreated files:")

for p in sorted(EXPORT_ROOT.rglob("*")):
    if p.is_file():
        print("-", p.relative_to(EXPORT_ROOT))